<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/May%2023_%20Arul_mcp_server_enterprise_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP Servers: Enterprise Use Case
## Building an Internal Supply Chain Inventory Management MCP Server

---

### What is MCP (Model Context Protocol)?

**MCP** is an open protocol (created by Anthropic) that standardizes how AI applications connect to external data sources and tools. Think of it as a **USB-C port for AI** — a universal interface that lets any AI model talk to any tool or data source.

#### Key Concepts:

| Concept | Description |
|---------|-------------|
| **MCP Server** | A lightweight program that exposes tools, resources, and prompts via the MCP protocol |
| **MCP Client** | The AI application (e.g., Claude Desktop, an IDE) that connects to servers |
| **Tools** | Functions the AI can call (like API endpoints) |
| **Resources** | Data the AI can read (like files or database records) |
| **Prompts** | Reusable prompt templates the server can offer |

#### Why MCP Matters for Enterprise:

1. **Standardized Integration** — One protocol to connect AI to all internal systems (ERP, CRM, ITSM, etc.)
2. **Security & Control** — Servers run inside your network; you control what the AI can access
3. **Separation of Concerns** — Domain experts build servers; AI teams build clients
4. **Auditability** — All tool calls are logged and traceable

---
## Enterprise Scenario: Supply Chain Inventory Management

**Imagine you're at a large manufacturing company.** Your team manages thousands of parts across multiple warehouses. Today, engineers use 3 different internal tools to:

- Check current stock levels
- Look up supplier information and lead times
- Place reorder requests when stock is low

**With an MCP server**, you can give an AI assistant access to all three capabilities through a single, secure interface. An engineer can simply ask:

> *"Are we running low on any titanium fasteners? If so, place a reorder with our preferred supplier."*

The AI uses the MCP server's tools to check inventory, evaluate thresholds, look up the supplier, and submit the order — all within the enterprise's security perimeter.

### Architecture

```
┌─────────────────────┐         MCP Protocol         ┌──────────────────────────┐
│                     │◄────── (stdio / SSE) ───────►│                          │
│   AI Application    │                              │   MCP Server:            │
│   (Claude Desktop,  │         Tools:               │   Inventory Management   │
│    IDE, Custom App)  │         • check_inventory    │                          │
│                     │         • get_supplier_info   │   Connects to:           │
│                     │         • place_reorder       │   • Inventory DB         │
│                     │         • get_low_stock_alert │   • Supplier API         │
│                     │                              │   • ERP System           │
└─────────────────────┘                              └──────────────────────────┘
```

---
## Step 1: Install Dependencies

The official MCP Python SDK makes it straightforward to build servers.

In [ ]:
!pip install mcp anthropic openai

---
## Step 2: Define the Enterprise Data Layer (Simulated)

In a real enterprise, these would connect to your ERP (SAP, Oracle), a warehouse management system, or internal APIs. Here we simulate them with in-memory data.

In [ ]:
from datetime import datetime, timedelta
import json

# --- Simulated Enterprise Data ---

INVENTORY_DB = {
    "TF-1001": {
        "name": "Titanium Fastener M8x30",
        "category": "Fasteners",
        "warehouse": "WH-EAST-01",
        "quantity_on_hand": 145,
        "reorder_point": 200,
        "unit_cost": 4.75,
        "currency": "USD",
        "last_counted": "2026-05-18",
    },
    "TF-1002": {
        "name": "Titanium Fastener M10x40",
        "category": "Fasteners",
        "warehouse": "WH-EAST-01",
        "quantity_on_hand": 520,
        "reorder_point": 300,
        "unit_cost": 6.20,
        "currency": "USD",
        "last_counted": "2026-05-18",
    },
    "AL-2050": {
        "name": "Aluminum Sheet 2mm 1200x2400",
        "category": "Raw Materials",
        "warehouse": "WH-WEST-03",
        "quantity_on_hand": 30,
        "reorder_point": 50,
        "unit_cost": 89.00,
        "currency": "USD",
        "last_counted": "2026-05-17",
    },
    "SS-3010": {
        "name": "Stainless Steel Rod 12mm",
        "category": "Raw Materials",
        "warehouse": "WH-EAST-01",
        "quantity_on_hand": 410,
        "reorder_point": 100,
        "unit_cost": 32.50,
        "currency": "USD",
        "last_counted": "2026-05-19",
    },
    "EP-4001": {
        "name": "Epoxy Resin Industrial Grade 5L",
        "category": "Chemicals",
        "warehouse": "WH-CHEM-02",
        "quantity_on_hand": 12,
        "reorder_point": 25,
        "unit_cost": 156.00,
        "currency": "USD",
        "last_counted": "2026-05-15",
    },
}

SUPPLIERS_DB = {
    "SUP-001": {
        "name": "TitaniumWorks Inc.",
        "contact_email": "orders@titaniumworks.example.com",
        "phone": "+1-555-0142",
        "categories": ["Fasteners", "Raw Materials"],
        "lead_time_days": 7,
        "rating": 4.8,
        "preferred": True,
        "minimum_order_value": 500.00,
    },
    "SUP-002": {
        "name": "Pacific Metals Corp.",
        "contact_email": "supply@pacificmetals.example.com",
        "phone": "+1-555-0198",
        "categories": ["Raw Materials"],
        "lead_time_days": 14,
        "rating": 4.2,
        "preferred": False,
        "minimum_order_value": 1000.00,
    },
    "SUP-003": {
        "name": "ChemSupply Global",
        "contact_email": "enterprise@chemsupply.example.com",
        "phone": "+1-555-0234",
        "categories": ["Chemicals"],
        "lead_time_days": 10,
        "rating": 4.5,
        "preferred": True,
        "minimum_order_value": 750.00,
    },
}

REORDER_LOG = []  # Stores submitted reorder requests

print("Enterprise data layer initialized.")
print(f"  • {len(INVENTORY_DB)} parts in inventory")
print(f"  • {len(SUPPLIERS_DB)} registered suppliers")

---
## Step 3: Build the MCP Server

This is the core of the example. We define an MCP server with **four tools** that an AI assistant can invoke:

| Tool | Purpose |
|------|--------|
| `check_inventory` | Look up current stock for a part by SKU or search by category |
| `get_low_stock_alerts` | Return all items below their reorder point |
| `get_supplier_info` | Look up supplier details, filtered by category |
| `place_reorder` | Submit a purchase order to restock a part |

Each tool has a **typed schema** so the AI knows exactly what parameters to pass.

In [ ]:
from mcp.server.fastmcp import FastMCP

# Initialize the MCP Server
mcp = FastMCP(
    name="InventoryManager",
    instructions="You are an enterprise inventory management assistant. "
    "Use the available tools to help engineers check stock levels, "
    "find supplier information, and place reorder requests.",
)


# --- Tool 1: Check Inventory ---
@mcp.tool()
def check_inventory(sku: str | None = None, category: str | None = None) -> str:
    """Check current inventory levels.

    Args:
        sku: Specific part SKU to look up (e.g., 'TF-1001')
        category: Filter by category (e.g., 'Fasteners', 'Raw Materials', 'Chemicals')

    Returns:
        JSON with inventory details including quantity, reorder point, and warehouse.
    """
    if sku:
        item = INVENTORY_DB.get(sku)
        if not item:
            return json.dumps({"error": f"SKU '{sku}' not found"})
        result = {sku: item}
    elif category:
        result = {
            k: v for k, v in INVENTORY_DB.items()
            if v["category"].lower() == category.lower()
        }
        if not result:
            return json.dumps({"error": f"No items found in category '{category}'"})
    else:
        result = INVENTORY_DB

    return json.dumps(result, indent=2)


# --- Tool 2: Get Low Stock Alerts ---
@mcp.tool()
def get_low_stock_alerts() -> str:
    """Get all inventory items that are below their reorder point.

    Returns:
        JSON list of items needing restock, with current quantity and deficit.
    """
    alerts = []
    for sku, item in INVENTORY_DB.items():
        if item["quantity_on_hand"] < item["reorder_point"]:
            alerts.append({
                "sku": sku,
                "name": item["name"],
                "category": item["category"],
                "warehouse": item["warehouse"],
                "quantity_on_hand": item["quantity_on_hand"],
                "reorder_point": item["reorder_point"],
                "deficit": item["reorder_point"] - item["quantity_on_hand"],
                "estimated_restock_cost": (
                    (item["reorder_point"] - item["quantity_on_hand"]) * item["unit_cost"]
                ),
            })

    if not alerts:
        return json.dumps({"message": "All items are above reorder thresholds."})

    return json.dumps({
        "alert_count": len(alerts),
        "total_restock_cost": sum(a["estimated_restock_cost"] for a in alerts),
        "items": alerts,
    }, indent=2)


# --- Tool 3: Get Supplier Info ---
@mcp.tool()
def get_supplier_info(
    category: str | None = None, preferred_only: bool = False
) -> str:
    """Look up supplier information.

    Args:
        category: Filter suppliers by product category they serve
        preferred_only: If True, only return preferred/primary suppliers

    Returns:
        JSON with supplier details including lead time, rating, and contact info.
    """
    results = {}
    for sup_id, sup in SUPPLIERS_DB.items():
        if category and category not in sup["categories"]:
            continue
        if preferred_only and not sup["preferred"]:
            continue
        results[sup_id] = sup

    if not results:
        return json.dumps({"error": "No suppliers match the given criteria"})

    return json.dumps(results, indent=2)


# --- Tool 4: Place Reorder ---
@mcp.tool()
def place_reorder(
    sku: str,
    supplier_id: str,
    quantity: int,
    urgency: str = "standard",
    notes: str = "",
) -> str:
    """Submit a purchase reorder request for a part.

    Args:
        sku: The part SKU to reorder
        supplier_id: The supplier to order from (e.g., 'SUP-001')
        quantity: Number of units to order
        urgency: Priority level — 'standard', 'expedited', or 'critical'
        notes: Optional notes for the procurement team

    Returns:
        JSON confirmation with order ID and estimated delivery date.
    """
    # Validate inputs
    if sku not in INVENTORY_DB:
        return json.dumps({"error": f"Unknown SKU: {sku}"})
    if supplier_id not in SUPPLIERS_DB:
        return json.dumps({"error": f"Unknown supplier: {supplier_id}"})
    if quantity <= 0:
        return json.dumps({"error": "Quantity must be positive"})
    if urgency not in ("standard", "expedited", "critical"):
        return json.dumps({"error": f"Invalid urgency: {urgency}"})

    item = INVENTORY_DB[sku]
    supplier = SUPPLIERS_DB[supplier_id]

    # Check minimum order value
    order_value = quantity * item["unit_cost"]
    if order_value < supplier["minimum_order_value"]:
        return json.dumps({
            "error": f"Order value ${order_value:.2f} is below supplier minimum of ${supplier['minimum_order_value']:.2f}"
        })

    # Calculate delivery estimate
    lead_days = supplier["lead_time_days"]
    if urgency == "expedited":
        lead_days = max(3, lead_days // 2)
    elif urgency == "critical":
        lead_days = max(2, lead_days // 3)

    estimated_delivery = (datetime.now() + timedelta(days=lead_days)).strftime("%Y-%m-%d")

    # Create the order
    order = {
        "order_id": f"PO-{len(REORDER_LOG) + 1:04d}",
        "sku": sku,
        "item_name": item["name"],
        "supplier_id": supplier_id,
        "supplier_name": supplier["name"],
        "quantity": quantity,
        "unit_cost": item["unit_cost"],
        "total_cost": order_value,
        "currency": item["currency"],
        "urgency": urgency,
        "estimated_delivery": estimated_delivery,
        "status": "submitted",
        "submitted_at": datetime.now().isoformat(),
        "notes": notes,
    }

    REORDER_LOG.append(order)

    return json.dumps({
        "success": True,
        "message": f"Reorder submitted successfully.",
        "order": order,
    }, indent=2)


print("MCP Server defined with 4 tools:")
print("  • check_inventory")
print("  • get_low_stock_alerts")
print("  • get_supplier_info")
print("  • place_reorder")

---
## Step 4: Test the Tools Directly

Before running the server, let's verify our tool logic works correctly by calling the functions directly.

In [ ]:
# Test: Check inventory for a specific SKU
print("=" * 60)
print("TEST: Check inventory for TF-1001")
print("=" * 60)
print(check_inventory(sku="TF-1001"))

In [ ]:
# Test: Get low stock alerts
print("=" * 60)
print("TEST: Low Stock Alerts")
print("=" * 60)
print(get_low_stock_alerts())

In [ ]:
# Test: Find preferred supplier for Fasteners
print("=" * 60)
print("TEST: Preferred suppliers for Fasteners")
print("=" * 60)
print(get_supplier_info(category="Fasteners", preferred_only=True))

In [ ]:
# Test: Place a reorder
print("=" * 60)
print("TEST: Place reorder for TF-1001")
print("=" * 60)
result = place_reorder(
    sku="TF-1001",
    supplier_id="SUP-001",
    quantity=200,
    urgency="expedited",
    notes="Stock critically low — production line at risk",
)
print(result)

---
## Step 5: Running the MCP Server

In production, you'd run the server as a standalone process. The MCP SDK supports two transport modes:

| Transport | Use Case |
|-----------|----------|
| **stdio** | Local integration (Claude Desktop, IDE plugins) — server runs as a subprocess |
| **SSE (Server-Sent Events)** | Remote/networked access — server runs as an HTTP service |

### Option A: Run via stdio (for Claude Desktop / local clients)

Save the server code to a file (e.g., `inventory_server.py`) and add this at the bottom:

```python
if __name__ == "__main__":
    mcp.run(transport="stdio")
```

Then configure your MCP client to run it. For **Claude Desktop**, add to `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "inventory-manager": {
      "command": "python",
      "args": ["/path/to/inventory_server.py"]
    }
  }
}
```

### Option B: Run via SSE (for remote/shared access)

```python
if __name__ == "__main__":
    mcp.run(transport="sse", host="0.0.0.0", port=8080)
```

This starts an HTTP server that multiple clients can connect to — ideal for shared enterprise deployments behind an API gateway.

---
## Step 6: Integrate with OpenAI GPT-4o-mini

Now we connect a **real LLM** — OpenAI's GPT-4o-mini — as the AI client that decides which MCP tools to call. This demonstrates the key MCP value proposition: **the same server tools work with any LLM that supports function calling.**

The flow:
1. We define our MCP tools as OpenAI-compatible function schemas
2. Send a user query to GPT-4o-mini along with the tool definitions
3. GPT-4o-mini decides which tools to call and with what arguments
4. We execute the tool calls against our MCP server functions
5. We send the results back to GPT-4o-mini for a final response

> **Note:** Set your `OPENAI_API_KEY` environment variable before running this cell, or replace the placeholder below.

In [ ]:
import os
from openai import OpenAI

# --- Initialize OpenAI Client ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here"))

# --- Define MCP Tools as OpenAI Function Schemas ---
# This is the bridge: MCP tool definitions → OpenAI function calling format

OPENAI_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check current inventory levels by SKU or category. Returns quantity on hand, reorder point, warehouse location, and unit cost.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {
                        "type": "string",
                        "description": "Specific part SKU to look up (e.g., 'TF-1001')",
                    },
                    "category": {
                        "type": "string",
                        "description": "Filter by category: 'Fasteners', 'Raw Materials', or 'Chemicals'",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_low_stock_alerts",
            "description": "Get all inventory items that are currently below their reorder point. Returns item details, deficit quantity, and estimated restock cost.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_supplier_info",
            "description": "Look up supplier information including contact details, lead time, rating, and minimum order value. Can filter by product category and preferred status.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "Filter suppliers by product category they serve",
                    },
                    "preferred_only": {
                        "type": "boolean",
                        "description": "If true, only return preferred/primary suppliers",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "place_reorder",
            "description": "Submit a purchase reorder request for a part. Validates inputs, checks minimum order value, and returns order confirmation with estimated delivery date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {
                        "type": "string",
                        "description": "The part SKU to reorder (e.g., 'TF-1001')",
                    },
                    "supplier_id": {
                        "type": "string",
                        "description": "The supplier ID to order from (e.g., 'SUP-001')",
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "Number of units to order",
                    },
                    "urgency": {
                        "type": "string",
                        "enum": ["standard", "expedited", "critical"],
                        "description": "Priority level for the order",
                    },
                    "notes": {
                        "type": "string",
                        "description": "Optional notes for the procurement team",
                    },
                },
                "required": ["sku", "supplier_id", "quantity"],
            },
        },
    },
]

# --- Map function names to our MCP tool implementations ---
TOOL_DISPATCH = {
    "check_inventory": check_inventory,
    "get_low_stock_alerts": get_low_stock_alerts,
    "get_supplier_info": get_supplier_info,
    "place_reorder": place_reorder,
}


def run_inventory_agent(user_query: str, verbose: bool = True) -> str:
    """
    Run a full agentic loop: send user query to GPT-4o-mini,
    let it call MCP tools, and return the final answer.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an enterprise inventory management assistant. "
                "You have access to tools for checking stock levels, finding suppliers, "
                "and placing reorder requests. Use the tools to answer the user's questions "
                "accurately. Always check current data before making recommendations. "
                "When placing orders, confirm the details in your response."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'=' * 70}")
        print(f"  USER QUERY: {user_query}")
        print(f"{'=' * 70}")

    # Agentic loop — keep going until the model stops calling tools
    iteration = 0
    max_iterations = 10

    while iteration < max_iterations:
        iteration += 1

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=OPENAI_TOOLS,
            tool_choice="auto",
        )

        choice = response.choices[0]

        # If the model wants to call tools
        if choice.finish_reason == "tool_calls" or choice.message.tool_calls:
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n[GPT-4o-mini] Calling tool: {func_name}({func_args})")

                # Execute the MCP tool
                tool_func = TOOL_DISPATCH[func_name]
                result = tool_func(**func_args)

                if verbose:
                    result_preview = result[:200] + "..." if len(result) > 200 else result
                    print(f"[Tool Result] {result_preview}")

                # Send tool result back to the model
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

        else:
            # Model is done — return the final response
            final_answer = choice.message.content
            if verbose:
                print(f"\n{'─' * 70}")
                print(f"[GPT-4o-mini] FINAL RESPONSE:")
                print(f"{'─' * 70}")
                print(final_answer)
            return final_answer

    return "Agent exceeded maximum iterations."


print("OpenAI GPT-4o-mini agent ready.")
print("Tools registered:", list(TOOL_DISPATCH.keys()))

---
## Step 7: Run the Agent — Example Queries

Now let's ask GPT-4o-mini real questions and watch it autonomously decide which MCP tools to call.

In [ ]:
# --- Query 1: Simple inventory check ---
run_inventory_agent("What's the current stock level of Titanium Fastener M8x30?")

In [ ]:
# --- Query 2: Multi-step reasoning — find low stock and recommend action ---
run_inventory_agent(
    "Are there any items below their reorder point? "
    "If so, find me the preferred supplier for each and recommend reorder quantities."
)

In [ ]:
# --- Query 3: Full autonomous workflow — the AI places an order ---
run_inventory_agent(
    "We have a critical production deadline next week. Check if Epoxy Resin is low, "
    "and if so, place a critical-urgency reorder with the preferred chemicals supplier. "
    "Order enough to get us 50% above the reorder threshold."
)

In [ ]:
# --- Query 4: Comparative question — requires checking multiple data sources ---
run_inventory_agent(
    "Compare all our raw materials suppliers. Which one has the best combination "
    "of lead time and rating? What's their minimum order value?"
)

In [ ]:
---
## Step 8: View the Audit Trail

Every tool call the LLM made resulted in a real action. Enterprise compliance requires this traceability.

In [ ]:
import pandas as pd

if REORDER_LOG:
    df = pd.DataFrame(REORDER_LOG)
    display_cols = ["order_id", "item_name", "supplier_name", "quantity",
                    "total_cost", "urgency", "estimated_delivery", "status"]
    print("\nReorder Audit Log (orders placed by GPT-4o-mini):")
    print(df[display_cols].to_string(index=False))
else:
    print("No orders have been placed yet. Run Query 3 above to see an order.")

---
## Key Takeaways

### What We Built
An MCP server integrated with **OpenAI GPT-4o-mini** as the AI client. The LLM autonomously decides which tools to call, chains multiple tool calls together, and synthesizes results into natural language responses.

### The MCP + LLM Integration Pattern

```
┌──────────────┐     function calling     ┌────────────────┐     tool execution     ┌──────────────┐
│  GPT-4o-mini │ ──────────────────────► │  Agent Loop     │ ──────────────────────► │  MCP Server  │
│  (reasoning) │ ◄────────────────────── │  (orchestrator) │ ◄────────────────────── │  (tools)     │
└──────────────┘     tool results         └────────────────┘     JSON responses      └──────────────┘
```

### Enterprise Benefits

| Benefit | How It Works |
|---------|-------------|
| **LLM-Agnostic** | Same MCP tools work with GPT-4o, Claude, Gemini, or local models |
| **Security** | LLM never gets direct DB access — only structured tool responses |
| **Governance** | Full audit trail of every action the AI took |
| **Human-in-the-Loop** | Add approval gates for high-value actions (e.g., orders > $5000) |
| **Observability** | Log every tool call, latency, and token usage |

### Production Considerations

1. **Authentication** — Add OAuth2/API keys for both the LLM API and MCP server
2. **Rate Limiting** — Cap tool calls per session to prevent runaway loops
3. **Approval Workflows** — For high-value orders, return "pending approval" instead of auto-submitting
4. **Cost Control** — GPT-4o-mini is cost-efficient (~$0.15/1M input tokens) for tool-heavy workflows
5. **Error Handling** — Add retries and graceful degradation for API failures
6. **Testing** — Use the MCP Inspector (`npx @modelcontextprotocol/inspector`) to test servers

### Next Steps

- Read the [MCP documentation](https://modelcontextprotocol.io)
- Explore the [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- Try swapping `gpt-4o-mini` for `gpt-4o` or Claude to compare tool-calling behavior
- Add **Resources** (expose inventory reports as readable documents)
- Add **Prompts** (reusable prompt templates like "daily inventory summary")
- Deploy the server via SSE and connect multiple AI clients simultaneously